# 비용의 나머지 절반: 에너지 -> cycle -> 랭킹

세 번째 탐색 노트북. 앞 두 노트북(`explore_scheduler`, `explore_blocks_and_cost`)이
입력/`_blocks`/`dram_bits`까지 다뤘으니, 여기선 **ipynb로 아직 안 본 나머지 전부**를 확인한다:

1. `energy_breakdown` - 어떤 버킷이 매핑을 가르고 어떤 게 고정인가
2. cycle 모델 - `actual_cycle = compute_work + fill + stall`
3. `_double_buffered` - prefetch 여유가 stall 을 어떻게 가르나
4. `optimize` / `report` - 전체 랭킹
5. `pareto_front` / `tradeoff` - 에너지 vs 사이클
6. `lpt_headroom` - 재정렬 여지 지표

각 `### 바꿔보기` 셀의 변수만 고치고 Shift+Enter.

In [ ]:
import os, sys, importlib, itertools
sys.path.insert(0, os.path.abspath("."))
import mxp_scheduler as s
importlib.reload(s)

# 공통 워크로드/하드웨어 (바꿔도 됨)
w  = s.Work(M=128, K=128, N=128,
            wbits=[[2,2,2,2],[4,4,4,4],[8,8,8,8],[2,8,2,8]], act_bits=8)
hw = s.HW(bank_size=1024, banks=32, dram_bw=64)
print("loaded:", s.__file__)
print("MT,KT,NT =", w.MT, w.KT, w.NT, "| cap_bits =", hw.cap_bits, "| eff_bw =", hw.eff_bw)

## 1. energy_breakdown - 뭐가 매핑을 가르고 뭐가 고정인가

`dram`/`onchip` 은 매핑따라 변하고, `mac`/`rmw` 는 GEMM 으로 고정 -> 랭킹에 무관.
두 루프순서를 나란히 비교해서 직접 확인.

In [ ]:
### 바꿔보기: 비교할 두 매핑
mA = s.Mapping(perm=("M","K","N"), m_in=1, k_in=1, n_in=4)   # spill 없음
mB = s.Mapping(perm=("N","K","M"), m_in=1, k_in=1, n_in=4)   # C spill

eA, eB = s.energy_breakdown(mA, w, hw), s.energy_breakdown(mB, w, hw)
print(f"{'bucket':<10}{''.join(mA.perm):>16}{''.join(mB.perm):>16}{'changes?':>11}")
for k in ("dram","onchip","mac","rmw","total"):
    print(f"{k:<10}{eA[k]:>16.0f}{eB[k]:>16.0f}{('VARIES' if eA[k]!=eB[k] else 'constant'):>11}")
print(f"\nmac_ops={s.mac_ops(w)} (M*K*N)   rmw_ops={s.rmw_ops(w)} (MT*KT*NT*1024*dispatch)")
print("=> mac/rmw 동일 -> 순위 못 바꿈. dram(+onchip) 만 매핑을 가른다.")

## 2. cycle 모델 - actual_cycle = compute_work + fill + stall

- `compute_work` : 순서 무관 이상적 계산량 (`32*NT*Sum(wbits)`)
- `fill` : 첫 블록 입력 페치 (시동, 못 가림)
- `stall` : 계산으로 못 가린 전송시간

여러 perm 의 세 항을 나란히 본다.

In [ ]:
cw = s.compute_work(w)
print(f"compute_work = {cw}  (모든 매핑 공통)\n")
print(f"{'perm':<6}{'compute':>10}{'fill':>10}{'stall':>12}{'actual_cycle':>14}{'double_buf':>12}")
for perm in itertools.permutations(("M","K","N")):
    m = s.Mapping(perm=perm, m_in=1, k_in=1, n_in=4)
    stall, fill = s.stall_fill(m, w, hw)
    db = s._double_buffered(m, w, hw)
    print(f"{''.join(perm):<6}{cw:>10}{fill:>10.1f}{stall:>12.1f}{s.actual_cycle(m,w,hw):>14.1f}{str(db):>12}")
print("\n# stall 이 작은 perm = 계산이 전송을 잘 가린 것. K 안쪽(no spill) 이 보통 유리.")

## 3. stall 이 어떻게 쌓이나 - 블록별 fetch vs hide

`_stall_of_order` 내부와 동일 로직을 블록마다 펼쳐서, 전송시간(`fetch/bw`)을
직전 계산(`hide`)이 얼마나 가리는지 본다. 못 가린 양이 그 블록 stall.

In [ ]:
### 바꿔보기: 한 매핑을 골라 블록별 stall 분해
m = s.Mapping(perm=("N","K","M"), m_in=1, k_in=1, n_in=4)
blocks = list(s._blocks(m, w))
bw = hw.eff_bw
db = s._double_buffered(m, w, hw, blocks)
print(f"perm={''.join(m.perm)}  bw={bw}  double_buffered={db}\n")
print(f"{'#':>2}{'idx':>10}{'fetch':>10}{'fetch/bw':>10}{'hide':>10}{'+stall':>10}")
seen=set(); prev=None; prev_compute=0.0; tot=0.0
for i,(idx,comp,a_blk,w_blk,c_blk) in enumerate(blocks):
    mn=(idx['M'],idx['N']); fetch=0.0
    if prev is not None:
        if (idx['K'],idx['N'])!=(prev['K'],prev['N']): fetch+=a_blk
        if (idx['M'],idx['K'])!=(prev['M'],prev['K']): fetch+=w_blk
        if mn!=(prev['M'],prev['N']):
            fetch+=c_blk
            if mn in seen: fetch+=c_blk
        hide = prev_compute if db else 0.0
        st = max(0.0, fetch/bw - hide); tot+=st
        print(f"{i:>2}{str((idx['M'],idx['K'],idx['N'])):>10}{fetch:>10.0f}{fetch/bw:>10.1f}{hide:>10.1f}{st:>10.1f}")
    seen.add(mn); prev=idx; prev_compute=comp
drain=blocks[-1][4]/bw; tot+=drain
print(f"trailing C drain: +{drain:.1f}")
print(f"-> stall 합 = {tot:.1f}   (stall_fill 의 stall = {s.stall_fill(m,w,hw)[0]:.1f}  일치 확인)")

## 4. double buffering 이 stall 을 가르는 순간

SRAM 용량(`cap_bits`)을 줄여가면 어느 지점에서 `_double_buffered` 가 False 로 꺾이고,
그 순간 stall 이 점프한다 (prefetch 불가 -> 페치 전부 노출). bank_size 를 쓸어본다.

In [ ]:
m = s.Mapping(perm=("M","K","N"), m_in=1, k_in=1, n_in=4)
blocks = list(s._blocks(m, w))
foot = s.footprint_bits(m, w)
a_blk, foot_w = blocks[0][2], max(b[3] for b in blocks)
print(f"footprint={foot}  double-buf 필요량 = footprint + a_blk + foot_w = {foot + a_blk + foot_w}\n")
print(f"{'bank_size':>10}{'cap_bits':>12}{'feasible':>10}{'double_buf':>12}{'stall':>12}")
for bsz in (1024, 256, 210, 200, 190, 180, 170):
    hw2 = s.HW(bank_size=bsz, banks=32, dram_bw=64)
    feas = s.feasible(m, w, hw2)
    if not feas:
        print(f"{bsz:>10}{hw2.cap_bits:>12}{'NO':>10}{'-':>12}{'(infeasible)':>12}")
        continue
    db = s._double_buffered(m, w, hw2)
    stall = s.stall_fill(m, w, hw2)[0]
    print(f"{bsz:>10}{hw2.cap_bits:>12}{'yes':>10}{str(db):>12}{stall:>12.1f}")
print("\n# double_buf True->False 로 꺾이는 순간 stall 이 확 뛴다 (계산이 전송을 못 가림).")

## 5. optimize + report - 전체 랭킹

`optimize` 가 feasible 후보를 `(energy, actual_cycle)` 로 정렬. 상위 10개 표.

In [ ]:
ranked = s.optimize(w, hw)
print(f"feasible 후보 수: {len(ranked)} / 전체 {len(s.gen_mappings(w))}\n")
print(s.report(ranked, w, hw))
print("\n1등:", ranked[0]['mapping'], " energy=", ranked[0]['energy'])

## 6. pareto_front + tradeoff - 에너지 vs 사이클

`pareto_front` = 지배당하지 않는 `(energy, cycle)` 곡선.
`tradeoff` = 그 양 끝 (OFF=최소에너지, ON=최速 중 최소에너지).

In [ ]:
front = s.pareto_front(ranked)
print(f"pareto front 점 개수: {len(front)}")
print(f"{'perm':<6}{'m_in':>5}{'k_in':>5}{'n_in':>5}{'energy':>16}{'actual_cycle':>14}")
for r in front:
    mm=r['mapping']
    print(f"{''.join(mm.perm):<6}{mm.m_in:>5}{mm.k_in:>5}{mm.n_in:>5}{r['energy']:>16.0f}{r['actual_cycle']:>14.1f}")
t = s.tradeoff(w, hw)
print(f"\nOFF(최소에너지): {t['off']['mapping']}  E={t['off']['energy']:.0f} cyc={t['off']['actual_cycle']:.0f}")
print(f"ON (최速최저E) : {t['on']['mapping']}  E={t['on']['energy']:.0f} cyc={t['on']['actual_cycle']:.0f}")

## 7. lpt_headroom - 블록 재정렬 여지 지표

자연 perm 순서 stall vs 블록을 계산량 내림차순(LPT)으로 재정렬했을 때 stall.
headroom > 0 이면 재정렬로 줄일 여지가 있다는 신호 (단 reuse 결합 때문에 LPT 가 항상 최적은 아님).

In [ ]:
for perm in [("M","K","N"), ("N","K","M"), ("K","M","N")]:
    m = s.Mapping(perm=perm, m_in=1, k_in=1, n_in=4)
    h = s.lpt_headroom(m, w, hw)
    print(f"perm={''.join(perm)}  natural={h['natural_stall']:.1f}  lpt={h['lpt_stall']:.1f}  headroom={h['headroom']:.1f}")